In [ ]:
##
## QLoRA Fine-Tuning of Gemma-4 31B for Redbox Message Tone (Foy & Sutherland)
## 3-class Negative/Contrast/Positive, US campaign redbox messages (N=910)
## 5-fold CV
## Author: Joan Timoneda
## Date: July 2026
##

import warnings
warnings.filterwarnings("ignore")
import os
os.environ["TRANSFORMERS_VERBOSITY"] = "error"

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, TaskType
import transformers
transformers.logging.set_verbosity_error()

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from collections import Counter
import json
import time
import gc


In [ ]:
# config

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Best open-weight decoder for this task: gemma few-shot, macro F1 0.9301 (DECODER_RESULTS_all; best overall)
DECODER_MODEL = "google/gemma-4-31b-it"
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRAD_ACCUM = 8
EPOCHS = 10
PATIENCE = 3
MAX_LEN = 512
N_FOLDS = 5
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
RESULTS_DIR = "results_qlora_workinghand_gemma31b"
os.makedirs(RESULTS_DIR, exist_ok=True)

# class names (reporting) and single-word answer proxies for label token prediction.
# The proxies appear in the prompt as the answer vocabulary and are verified single
# tokens for this model's tokenizer (leading-space encoding).
CLASS_LABELS = ["Negative", "Contrast", "Positive"]
LABEL_PROXIES = ["Negative", "Contrast", "Positive"]

N_CLASSES = len(CLASS_LABELS)

print(f"Model: {DECODER_MODEL}")
print(f"Classes: {N_CLASSES} {CLASS_LABELS}")
print(f"LoRA r={LORA_R}, alpha={LORA_ALPHA}")
print(f"Batch: {BATCH_SIZE}, grad accum: {GRAD_ACCUM}, effective batch: {BATCH_SIZE * GRAD_ACCUM}")


In [ ]:
# load model with 4-bit quantization

print(f"Loading {DECODER_MODEL}...")

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    DECODER_MODEL,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Base model loaded.")

# check label proxy tokens
print("\nLabel token check:")
for full_name, proxy in zip(CLASS_LABELS, LABEL_PROXIES):
    ids = tokenizer.encode(" " + proxy, add_special_tokens=False)
    status = '<-- single token' if len(ids) == 1 else f'MULTI ({len(ids)} tokens) -- FIX THIS'
    print(f"  {full_name:15s} -> '{proxy}': {ids} {status}")


In [ ]:
# LoRA

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type=TaskType.CAUSAL_LM,
    bias="none"
)

# test LoRA param count
test_model = get_peft_model(base_model, lora_config)
trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in test_model.parameters())
print(f"LoRA trainable params: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"Total params: {total:,}")

del test_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# load data

df = pd.read_excel("../../../data/foy_sutherland/redbox_text_labels.xlsx")

label_map = {"Negative": 0, "Contrast": 1, "Positive": 2}
texts = [str(t) for t in df["text"].tolist()]
labels = [label_map[l] for l in df["label_main"].tolist()]

print(f"Loaded {len(texts)} redbox messages")
print(f"\nLabel distribution:")
for label, count in df["label_main"].value_counts().items():
    print(f"  {label:12s}: {count}")


In [ ]:
# prompt and dataset class

def build_prompt(text):
    return f'Classify the tone of this "red box" message, an instruction a U.S. congressional campaign posts on its website to signal to allied super PACs what advertising to run. Answer Negative if the message refers only to an opponent, presenting them unfavorably or attacking them. Answer Contrast if the message refers to both the favored candidate and an opponent, comparing them. Answer Positive if the message refers only to the favored or sponsoring candidate, presenting them favorably.\nMessage: "{text}"\nAnswer:'

class TaskDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, label_proxies, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.label_proxies = label_proxies
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        label_proxy = self.label_proxies[label]

        prompt = build_prompt(text)

        prompt_tokens = self.tokenizer(prompt, add_special_tokens=False, truncation=True, max_length=self.max_len - 5)
        # leading space so train-time label ids match the " " + proxy ids scored at
        # inference (bare vs spaced words tokenize differently in BPE vocabs)
        label_tokens = self.tokenizer(" " + label_proxy, add_special_tokens=False)

        input_ids = prompt_tokens["input_ids"] + label_tokens["input_ids"]
        attention_mask = [1] * len(input_ids)
        labels = [-100] * len(prompt_tokens["input_ids"]) + label_tokens["input_ids"]

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

def collate_fn(batch):
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = []
    attention_mask = []
    labels = []

    for item in batch:
        pad_len = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [tokenizer.pad_token_id] * pad_len)
        attention_mask.append(item["attention_mask"] + [0] * pad_len)
        labels.append(item["labels"] + [-100] * pad_len)

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

# sanity check
test_ds = TaskDataset(texts[:3], labels[:3], tokenizer, LABEL_PROXIES, MAX_LEN)
sample = test_ds[0]
decoded = tokenizer.decode(sample["input_ids"][-15:])
label_toks = [x for x in sample["labels"] if x != -100]
print(f"Sample ends with: ...{decoded}")
print(f"Sample label tokens: {label_toks}")
print(f"Decoded label: {tokenizer.decode(label_toks)}")
print(f"True label: {CLASS_LABELS[labels[0]]}")
del test_ds


In [ ]:
# inference function

def classify_qlora(texts, model, tokenizer, label_proxies, max_len=512):
    model.eval()
    preds = []

    label_token_ids = {}
    for i, name in enumerate(label_proxies):
        ids = tokenizer.encode(" " + name, add_special_tokens=False)
        label_token_ids[i] = ids[0]

    for i in range(len(texts)):
        prompt = build_prompt(texts[i])
        tokens = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_len).to(device)

        with torch.no_grad():
            outputs = model(**tokens)

        logits = outputs.logits[0, -1, :]
        scores = [logits[label_token_ids[c]].item() for c in range(N_CLASSES)]
        pred = int(np.argmax(scores))
        preds.append(pred)

        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(texts)}", end="\r")

    return preds


In [ ]:
# 5-fold CV

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(texts, labels)):

    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}')
    print(f'{"="*60}')

    train_texts_fold = [texts[i] for i in train_idx]
    train_labels_fold = [labels[i] for i in train_idx]
    val_texts_fold = [texts[i] for i in val_idx]
    val_labels_fold = [labels[i] for i in val_idx]

    print(f'  Train: {len(train_texts_fold)} | Val: {len(val_texts_fold)}')

    # fresh LoRA adapter each fold
    model = get_peft_model(base_model, lora_config)
    model.train()

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE
    )

    train_dataset = TaskDataset(train_texts_fold, train_labels_fold, tokenizer, LABEL_PROXIES, MAX_LEN)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

    print(f"  Train batches: {len(train_loader)}")

    best_f1 = 0
    best_state = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        print(f'\n  ======== Epoch {epoch+1} / {EPOCHS} ========')
        print('  Training...')

        model.train()
        epoch_loss = 0
        n_batches = 0
        optimizer.zero_grad()

        for batch_i, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            batch_labels = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=batch_labels
            )
            loss = outputs.loss / GRAD_ACCUM
            loss.backward()

            if (batch_i + 1) % GRAD_ACCUM == 0 or (batch_i + 1) == len(train_loader):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()

            epoch_loss += outputs.loss.item()
            n_batches += 1

            if n_batches % 50 == 0:
                elapsed = time.time() - t0
                mins, secs = divmod(int(elapsed), 60)
                print(f'    Batch {n_batches}/{len(train_loader)}  |  Loss: {epoch_loss/n_batches:.4f}  |  Elapsed: {mins}m{secs:02d}s', end='\r')

        avg_loss = epoch_loss / n_batches
        epoch_time = time.time() - t0
        mins, secs = divmod(int(epoch_time), 60)
        print(f'\n    Avg loss: {avg_loss:.4f}  |  Epoch time: {mins}m{secs:02d}s')

        # validate
        print('  Validating...')
        val_preds = classify_qlora(val_texts_fold, model, tokenizer, LABEL_PROXIES, MAX_LEN)
        val_f1_macro = f1_score(val_labels_fold, val_preds, average="macro", zero_division=0)
        val_f1_weighted = f1_score(val_labels_fold, val_preds, average="weighted", zero_division=0)

        print(f'    Val Macro F1: {val_f1_macro:.4f}  |  Weighted F1: {val_f1_weighted:.4f}')

        if val_f1_macro > best_f1:
            best_f1 = val_f1_macro
            best_state = {k: v.clone() for k, v in model.state_dict().items() if "lora" in k}
            patience_counter = 0
            print(f'    ** New best model (F1: {best_f1:.4f}) **')
        else:
            patience_counter += 1
            print(f'    No improvement ({patience_counter}/{PATIENCE})')

        if patience_counter >= PATIENCE and epoch >= 2:
            print(f'    Early stopping at epoch {epoch+1}.')
            break

    # final eval with best weights
    for k, v in best_state.items():
        if k in model.state_dict():
            model.state_dict()[k].copy_(v)

    print('\n  Final evaluation...')
    val_preds = classify_qlora(val_texts_fold, model, tokenizer, LABEL_PROXIES, MAX_LEN)
    f1_macro = f1_score(val_labels_fold, val_preds, average="macro", zero_division=0)
    f1_weighted = f1_score(val_labels_fold, val_preds, average="weighted", zero_division=0)

    # per-class F1
    per_class_f1 = f1_score(val_labels_fold, val_preds, average=None, labels=list(range(N_CLASSES)), zero_division=0)

    fold_metrics = {
        "fold": fold + 1,
        "best_epoch": int(epoch + 1 - patience_counter),
        "train_loss": round(avg_loss, 4),
        "f1_macro": round(f1_macro, 4),
        "f1_weighted": round(f1_weighted, 4),
    }
    for i, label in enumerate(CLASS_LABELS):
        fold_metrics[f"f1_{label}"] = round(per_class_f1[i], 4)

    fold_results.append(fold_metrics)

    print(f"  Macro F1:    {f1_macro:.4f}")
    print(f"  Weighted F1: {f1_weighted:.4f}")
    print(f"\n  Per-class F1:")
    for i, label in enumerate(CLASS_LABELS):
        print(f"    {label:15s}: {per_class_f1[i]:.3f}")

    # save running results
    pd.DataFrame(fold_results).to_csv(f"{RESULTS_DIR}/cv_results_running.csv", index=False)
    print(f"\n  Saved fold {fold+1}.")

    # unload LoRA for next fold
    del model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# summary

results_df = pd.DataFrame(fold_results)
results_df.to_csv(f"{RESULTS_DIR}/cv_results_final.csv", index=False)

print("=== QLoRA Gemma-4 31B: FOY-SUTHERLAND REDBOX TONE CV RESULTS ===")
print(f"Macro F1:    {results_df['f1_macro'].mean():.4f} (SD: {results_df['f1_macro'].std():.4f})")
print(f"Weighted F1: {results_df['f1_weighted'].mean():.4f} (SD: {results_df['f1_weighted'].std():.4f})")
print()
print("Per-class F1 (mean across folds):")
for label in CLASS_LABELS:
    col = f"f1_{label}"
    print(f"  {label:15s}: {results_df[col].mean():.3f} (SD: {results_df[col].std():.3f})")
print()
print(results_df[['fold', 'best_epoch', 'train_loss', 'f1_macro', 'f1_weighted']].to_string(index=False))
